# 💰 Notebook 1 — Data Cleaning & Preparation

We load the raw transaction and budget files, clean them, enrich them with
derived columns, and export a clean dataset for all subsequent notebooks.

**Datasets:**
- `transactions.csv` — 69 income and expense transactions (Jan–Mar 2024)
- `budgets.csv` — Monthly budget targets per category

**Person profile:** Single person, Johannesburg, R28,000 salary + freelance side income

In [ ]:
import pandas as pd
import numpy as np
import os
print('Libraries loaded ✅')

## Step 1 — Load Raw Data

In [ ]:
tx  = pd.read_csv('../data/transactions.csv')
bud = pd.read_csv('../data/budgets.csv')
print('Transactions shape:', tx.shape)
print('Budget rows:', len(bud))
tx.head(10)

In [ ]:
bud

## Step 2 — Inspect for Issues

Check for missing values, wrong data types, and any unexpected values before cleaning.

In [ ]:
print('Data types:')
print(tx.dtypes)
print()
print('Missing values:')
print(tx.isnull().sum())
print()
print('Unique types:', tx['type'].unique())
print('Unique categories:', sorted(tx['category'].unique()))

## Step 3 — Clean and Enrich

Parse dates properly and add derived columns for month, week, and day name. These enable time-based grouping in later analysis.

In [ ]:
tx['date']       = pd.to_datetime(tx['date'])
tx['month']      = tx['date'].dt.month
tx['month_name'] = tx['date'].dt.month_name()
tx['week']       = tx['date'].dt.to_period('W').astype(str)
tx['day_name']   = tx['date'].dt.day_name()
tx['notes']      = tx['notes'].fillna('').str.strip()

print('Date range:', tx['date'].min().date(), '→', tx['date'].max().date())
tx[['date','month','month_name','day_name','type','category','amount']].head(10)

## Step 4 — Split Income vs Expenses

In [ ]:
income   = tx[tx['type'] == 'Income'].copy()
expenses = tx[tx['type'] == 'Expense'].copy()

print(f'Income transactions  : {len(income)}')
print(f'Expense transactions : {len(expenses)}')
print()
print('Income categories:', income['category'].unique())
print('Expense categories:', sorted(expenses['category'].unique()))

## Step 5 — Monthly Summary

Calculate total income, salary, side income, total expenses, net savings, and savings rate for each month.

In [ ]:
monthly_income   = income.groupby(['month','month_name'])['amount'].sum().reset_index().rename(columns={'amount':'total_income'})
monthly_salary   = income[income['category']=='Salary'].groupby(['month','month_name'])['amount'].sum().reset_index().rename(columns={'amount':'salary'})
monthly_side     = income[income['category']=='Side Income'].groupby(['month','month_name'])['amount'].sum().reset_index().rename(columns={'amount':'side_income'})
monthly_expenses = expenses.groupby(['month','month_name'])['amount'].sum().reset_index().rename(columns={'amount':'total_expenses'})

monthly = monthly_income.merge(monthly_salary, on=['month','month_name'], how='left')
monthly = monthly.merge(monthly_side,          on=['month','month_name'], how='left')
monthly = monthly.merge(monthly_expenses,      on=['month','month_name'], how='left')
monthly['side_income']  = monthly['side_income'].fillna(0)
monthly['net_savings']  = monthly['total_income'] - monthly['total_expenses']
monthly['savings_rate'] = (monthly['net_savings'] / monthly['total_income'] * 100).round(2)
monthly = monthly.sort_values('month')

print('Monthly Financial Summary:')
monthly

## Step 6 — Budget vs Actual per Category per Month

In [ ]:
cat_monthly = expenses.groupby(['month','month_name','category'])['amount'].sum().reset_index().rename(columns={'amount':'actual'})
cat_monthly = cat_monthly.merge(bud, on='category', how='left')
cat_monthly['variance']     = cat_monthly['actual'] - cat_monthly['monthly_budget']
cat_monthly['over_budget']  = cat_monthly['variance'] > 0
cat_monthly['pct_of_budget']= (cat_monthly['actual'] / cat_monthly['monthly_budget'] * 100).round(2)

print('Over-budget entries:')
cat_monthly[cat_monthly['over_budget']].sort_values('variance', ascending=False)

## Step 7 — Export Clean Data

In [ ]:
tx.to_csv('../data/cleaned.csv', index=False)
print('✅ Exported to data/cleaned.csv')
print(f'   Rows: {len(tx)}')
tx.describe()